In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import urllib.request
from PIL import Image
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

FEATURES_PATH = r"D:\creator-analytics\clean_data\thumbnail_features.parquet"
VIDEOS_PATH   = r"D:\creator-analytics\clean_data\yt_videos_with_local_transcripts.parquet"
HEADERS       = {"User-Agent": "Mozilla/5.0"}

In [ ]:
feats  = pd.read_parquet(FEATURES_PATH)
videos = pd.read_parquet(VIDEOS_PATH, columns=[
    "videoId", "title", "channelTitle", "thumbnail_maxres_url",
    "thumbnail_high_url", "is_short", "viewCount"
])

df = feats.merge(videos, left_on="video_id", right_on="videoId", how="left")
df["thumb_url"] = df["thumbnail_maxres_url"].fillna(df["thumbnail_high_url"])
df["viewCount"] = pd.to_numeric(df["viewCount"], errors="coerce")

print(f"{len(df):,} rows, {df['thumb_url'].notna().sum():,} with thumbnail URL")

In [ ]:
# Stratified sample: 10 per face/text combination
rng = np.random.default_rng(42)

strata = [
    (True,  True,  "face + text"),
    (True,  False, "face only"),
    (False, True,  "text only"),
    (False, False, "neither"),
]

parts = []
for face, text, label in strata:
    pool = df[(df["face_present"] == face) & (df["text_present"] == text) & df["thumb_url"].notna()]
    n    = min(10, len(pool))
    s    = pool.sample(n, random_state=42)
    s    = s.copy()
    s["stratum"] = label
    parts.append(s)
    print(f"{label}: {n} sampled (pool={len(pool)})")

sample = pd.concat(parts, ignore_index=True)

In [ ]:
def fetch_image(url, timeout=10):
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return Image.open(BytesIO(r.read())).convert("RGB")
    except Exception:
        return None


def feature_text(row):
    lines = [
        f"clip_align  : {row.clip_title_align:.3f}" if pd.notna(row.clip_title_align) else "clip_align  : —",
        f"brightness  : {row.brightness_mean:.2f}   saturation: {row.saturation_mean:.2f}",
        f"text_words  : {int(row.text_word_count)}   has_number: {row.text_has_number}",
        f"face_count  : {int(row.face_count)}   area: {row.face_area_ratio:.3f}   frontal: {row.face_is_frontal}",
        f"yaw: {row.face_yaw_deg:+.1f}°   pitch: {row.face_pitch_deg:+.1f}°",
        f"mouth_open  : {row.mouth_openness:.3f}   eye_open: {row.eye_openness_mean:.3f}",
        f"gaze_off_cam: {row.gaze_off_camera}",
        f"views       : {int(row.viewCount):,}" if pd.notna(row.viewCount) else "views       : —",
    ]
    return "\n".join(lines)

In [ ]:
COLS = 5

for stratum_label in [s[2] for s in strata]:
    group = sample[sample["stratum"] == stratum_label].reset_index(drop=True)
    n     = len(group)
    rows  = int(np.ceil(n / COLS))

    fig = plt.figure(figsize=(COLS * 4, rows * 5.5))
    fig.suptitle(stratum_label.upper(), fontsize=14, fontweight="bold", y=1.01)

    for idx, row in group.iterrows():
        ax = fig.add_subplot(rows, COLS, idx + 1)
        img = fetch_image(row["thumb_url"])
        if img:
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "(no image)", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

        title  = str(row.get("title", ""))[:45] + ("…" if len(str(row.get("title", ""))) > 45 else "")
        ch     = str(row.get("channelTitle", ""))[:25]
        feats_ = feature_text(row)
        ax.set_title(f"{ch}\n{title}", fontsize=7, pad=3)
        ax.text(0, -0.02, feats_, transform=ax.transAxes,
                fontsize=6, va="top", family="monospace",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#f5f5f5", edgecolor="#cccccc"))

    plt.tight_layout()
    plt.show()